In [1]:
import cv2
from ultralytics import YOLO

# Load the YOLOv8n model
model = YOLO('yolov8n.pt')

# Global speed variable
speed = 100  # Initial speed of the vehicle


def detect_objects(frame):
    """Perform object detection, return annotated frame, and brake status."""
    global speed
    results = model(frame)
    annotated_frame = frame.copy()
    brake_message = None
    should_brake = False

    # Define thresholds for reducing speed based on object proximity
    brake_threshold = 300  # Object height threshold to apply brake
    slow_down_threshold = 150  # Object height threshold to slow down

    # Iterate over each detected result
    for result in results:
        for box in result.boxes:
            confidence = float(box.conf.cpu().numpy())  # Confidence score
            class_id = int(box.cls.cpu().numpy())  # Class ID as integer
            if confidence > 0.5:  # Only consider detections with confidence > 0.5
                # Get bounding box coordinates and the class label
                x1, y1, x2, y2 = map(int, box.xyxy[0].cpu().numpy())
                label = model.names[class_id]

                # Draw bounding box and label
                cv2.rectangle(annotated_frame, (x1, y1),
                              (x2, y2), (0, 255, 0), 2)
                label_text = f"{label} ({confidence:.2f})"
                cv2.putText(annotated_frame, label_text, (x1, y1 - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

                # Adjust speed or apply brake based on object proximity
                if label in ['person', 'cat', 'vehicle', 'obstacle', 'traffic signal', 'stop sign']:
                    box_height = y2 - y1

                    if box_height > brake_threshold:
                        should_brake = True
                    elif box_height > slow_down_threshold:
                        # Slow down but keep a minimum of 30 km/h
                        speed = max(speed - 20, 30)

    if should_brake:
        brake_message = "Pressing brake! Object is too close!"
        speed = 0  # Apply full brake
    else:
        brake_message = f"Speed: {speed} km/h"
        speed = min(speed + 5, 100)  # Gradually increase speed to max 100 km/h

    return annotated_frame, brake_message


def start_camera():
    """Initialize and start the camera feed."""
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("Error: Could not open the webcam.")
        return

    while True:
        ret, frame = cap.read()
        if not ret:
            print("Error: Could not read frame.")
            break

        # Detect objects and annotate frame
        detected_frame, message = detect_objects(frame)

        # Display speed/brake message
        cv2.putText(detected_frame, message, (50, 50),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

        # Show the frame
        cv2.imshow('YOLOv8 Object Detection with Speed Control', detected_frame)

        # Break the loop if 'q' is pressed
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()


# Run the camera feed with object detection
start_camera()


0: 480x640 7 persons, 1 bicycle, 14 cars, 23 motorcycles, 3 airplanes, 15 buss, 2 trains, 1 truck, 72 boats, 4 traffic lights, 4 fire hydrants, 52 stop signs, 18 parking meters, 1 bench, 1 horse, 18 bears, 1 zebra, 34 umbrellas, 2 handbags, 6 suitcases, 2 frisbees, 2 sports balls, 1 surfboard, 2 bottles, 2 beds, 11 dining tables, 1 sink, 396.2ms
Speed: 2.0ms preprocess, 396.2ms inference, 15.0ms postprocess per image at shape (1, 3, 480, 640)



C:\Users\VENKATA REDDY\AppData\Local\Temp\ipykernel_58972\4148438981.py:26: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  confidence = float(box.conf.cpu().numpy())  # Confidence score
C:\Users\VENKATA REDDY\AppData\Local\Temp\ipykernel_58972\4148438981.py:27: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  class_id = int(box.cls.cpu().numpy())  # Class ID as integer


0: 480x640 9 persons, 3 bicycles, 14 cars, 27 motorcycles, 1 airplane, 2 buss, 7 trains, 2 trucks, 35 boats, 2 traffic lights, 6 fire hydrants, 64 stop signs, 6 parking meters, 3 dogs, 7 horses, 1 elephant, 10 bears, 16 umbrellas, 41 handbags, 8 ties, 5 suitcases, 4 frisbees, 4 bottles, 1 knife, 1 banana, 1 couch, 4 beds, 11 dining tables, 3 remotes, 1 sink, 1 teddy bear, 404.4ms
Speed: 2.9ms preprocess, 404.4ms inference, 13.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 bicycle, 2 cars, 1 motorcycle, 2 airplanes, 1 bus, 23 trains, 8 stop signs, 4 parking meters, 3 birds, 1 cat, 5 dogs, 10 horses, 62 bears, 1 giraffe, 10 backpacks, 70 umbrellas, 11 handbags, 17 ties, 6 frisbees, 8 snowboards, 2 sports balls, 1 surfboard, 1 bottle, 23 knifes, 10 spoons, 1 banana, 2 oranges, 5 cakes, 1 bed, 1 dining table, 2 remotes, 2 sinks, 3 toothbrushs, 392.2ms
Speed: 2.2ms preprocess, 392.2ms inference, 10.6ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 6 persons,